# HealthBot: AI-Powered Patient Education System

A LangGraph-based prototype built for MediTech Solutions. The bot:

1. Asks the patient for a health topic.
2. Searches Tavily for up-to-date medical information.
3. Summarizes the results in patient-friendly language.
4. Presents the summary and lets the patient read it.
5. Quizzes the patient with one comprehension question based only on the summary.
6. Grades the answer with a letter grade and a citation-backed justification.
7. Lets the patient learn about another topic (with state reset for privacy) or exit.

## 1. Setup: load environment variables

Keys are loaded from `.env` in the project folder (the assignment refers to this file as `config.env` -- here it's simply named `.env`).

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

assert os.getenv("TAVILY_API_KEY") is not None, "TAVILY_API_KEY missing from .env"

# At least one LLM provider key must be present. The model selector below will
# only offer providers whose keys are actually set.
_has_openai = bool(os.getenv("OPENAI_API_KEY"))
_has_google = bool(os.getenv("GOOGLE_API_KEY"))
_has_anthropic = bool(os.getenv("ANTHROPIC_API_KEY"))

assert any([_has_openai, _has_google, _has_anthropic]), (
    "No LLM provider API key found. Set one of OPENAI_API_KEY, GOOGLE_API_KEY, "
    "or ANTHROPIC_API_KEY in .env"
)

print("Tavily key loaded.")
print("OpenAI key present:", _has_openai)
print("Google key present:", _has_google)
print("Anthropic key present:", _has_anthropic)

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from typing import Annotated, Optional, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## 3. Model selector

Choose which LLM provider/model drives the HealthBot for this session. Only providers with a valid key in `.env` are offered. All providers are wrapped behind the same `BaseChatModel` interface, so every node downstream works identically regardless of which one is picked.

In [ ]:
AVAILABLE_MODELS = {}

if _has_openai:
    AVAILABLE_MODELS["1"] = {
        "label": "OpenAI - gpt-4o-mini",
        "provider": "openai",
        "model": "gpt-4o-mini",
    }
if _has_google:
    AVAILABLE_MODELS["2"] = {
        "label": "Google Gemini - gemini-2.5-flash",
        "provider": "google_genai",
        "model": "gemini-2.5-flash",
    }
if _has_anthropic:
    AVAILABLE_MODELS["3"] = {
        "label": "Anthropic - claude-3-5-sonnet-latest",
        "provider": "anthropic",
        "model": "claude-3-5-sonnet-latest",
    }


def build_chat_model(provider: str, model: str):
    """Instantiate a chat model for the given provider using the appropriate
    LangChain integration package. Returns a BaseChatModel."""
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, api_key=os.getenv("OPENAI_API_KEY"), temperature=0)
    if provider == "google_genai":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model, api_key=os.getenv("GOOGLE_API_KEY"), temperature=0)
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model, api_key=os.getenv("ANTHROPIC_API_KEY"), temperature=0)
    raise ValueError(f"Unknown provider: {provider}")


def select_model():
    print("Select the LLM that will power this HealthBot session:")
    for key, info in AVAILABLE_MODELS.items():
        print(f"  {key}. {info['label']}")

    choice = input(f"Enter a number ({'/'.join(AVAILABLE_MODELS.keys())}): ").strip()
    while choice not in AVAILABLE_MODELS:
        choice = input("Please enter a valid option number: ").strip()

    picked = AVAILABLE_MODELS[choice]
    print(f"\nUsing {picked['label']} for this session.\n")
    return build_chat_model(picked["provider"], picked["model"])


llm = select_model()

## 4. Tavily search tool

Using the LangChain community tool for Tavily, per the project spec. `include_domains` is left open, but the search prompt instructs the model to prefer reputable medical sources (e.g. NIH, CDC, Mayo Clinic, MedlinePlus, WHO).

In [ ]:
tavily_tool = TavilySearchResults(max_results=5)

# Bind the tool to the LLM and *force* a tool call (tool_choice="any") so the
# model is required to search rather than answer from its own knowledge.
llm_with_tavily = llm.bind_tools([tavily_tool], tool_choice="any")

## 5. State schema

A single `HealthBotState` TypedDict is threaded through every node. `messages` accumulates the full conversation (including tool calls/results) via LangGraph's `add_messages` reducer, so the model always has access to prior context. Topic-specific fields (`topic`, `search_results`, `summary`, `quiz_question`, `patient_answer`, `grade`, `feedback`) are explicitly cleared by the `reset_state` node whenever the patient starts a new topic, to keep information from one patient session from leaking into the next.

In [ ]:
class HealthBotState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: Optional[str]
    search_results: Optional[str]
    summary: Optional[str]
    quiz_question: Optional[str]
    patient_answer: Optional[str]
    grade: Optional[str]
    feedback: Optional[str]
    continue_session: Optional[bool]

## 6. Nodes

Each node has a single responsibility: collecting input, calling the model, or displaying output.

In [ ]:
# ---------------------------------------------------------------------------
# Node: ask_topic - collect the health topic from the patient
# ---------------------------------------------------------------------------
def ask_topic(state: HealthBotState) -> HealthBotState:
    topic = input("\nWhat health topic or medical condition would you like to learn about? ").strip()
    print(f"\nGot it -- let's learn about: {topic}\n")
    return {
        "topic": topic,
        "messages": [HumanMessage(content=f"I would like to learn about: {topic}")],
    }

In [ ]:
# ---------------------------------------------------------------------------
# Node: search_topic - have the LLM call Tavily for the topic
#
# Hybrid approach: the LLM is required (tool_choice="any") to call the Tavily
# tool. If, for any reason, the model does not return a tool call, we fall
# back to calling Tavily directly with the topic as the query -- guaranteeing
# a search always happens.
# ---------------------------------------------------------------------------
SEARCH_SYSTEM_PROMPT = (
    "You are a medical research assistant. Use the tavily_search_results_json tool "
    "to search for clear, up-to-date, and reputable medical information (such as from "
    "sources like the NIH, CDC, Mayo Clinic, MedlinePlus, or WHO) about the patient's "
    "requested health topic. Formulate a focused search query for the topic."
)


def search_topic(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]

    ai_message = llm_with_tavily.invoke(
        [SystemMessage(content=SEARCH_SYSTEM_PROMPT), HumanMessage(content=f"Topic: {topic}")]
    )

    tool_messages = []
    if ai_message.tool_calls:
        for call in ai_message.tool_calls:
            query = call["args"].get("query", topic)
            results = tavily_tool.invoke(query)
            tool_messages.append(
                ToolMessage(content=str(results), tool_call_id=call["id"])
            )
    else:
        # Deterministic fallback: force a direct Tavily search on the topic.
        results = tavily_tool.invoke(topic)
        tool_messages.append(ToolMessage(content=str(results), tool_call_id="fallback"))

    search_results_text = "\n\n".join(tm.content for tm in tool_messages)

    return {
        "search_results": search_results_text,
        "messages": [ai_message, *tool_messages],
    }

In [ ]:
# ---------------------------------------------------------------------------
# Node: summarize_results - summarize Tavily results in patient-friendly language
# ---------------------------------------------------------------------------
SUMMARY_SYSTEM_PROMPT = (
    "You are HealthBot, a patient education assistant. You will be given raw search "
    "results about a health topic. Write a clear, empathetic, patient-friendly summary "
    "of 3 to 4 paragraphs.\n\n"
    "Rules:\n"
    "- Use ONLY the information contained in the provided search results. Do not add "
    "facts from your own general knowledge or any other source.\n"
    "- Avoid unexplained medical jargon; explain terms simply.\n"
    "- Do not give personalized medical advice or diagnoses -- this is general "
    "educational information only.\n"
    "- Write in plain prose paragraphs (no bullet lists, no headers)."
)


def summarize_results(state: HealthBotState) -> HealthBotState:
    prompt = (
        f"Health topic: {state['topic']}\n\n"
        f"Search results:\n{state['search_results']}\n\n"
        "Please write the 3-4 paragraph patient-friendly summary now."
    )
    response = llm.invoke([SystemMessage(content=SUMMARY_SYSTEM_PROMPT), HumanMessage(content=prompt)])
    summary = response.content

    return {"summary": summary, "messages": [response]}

In [ ]:
# ---------------------------------------------------------------------------
# Node: present_summary - show the summary and let the patient read it
# ---------------------------------------------------------------------------
def present_summary(state: HealthBotState) -> HealthBotState:
    print("=" * 70)
    print(f"HERE'S WHAT WE FOUND ABOUT: {state['topic'].upper()}")
    print("=" * 70)
    print(state["summary"])
    print("=" * 70)
    input("\nTake your time reading the summary above. Press Enter when you're ready for a quick comprehension check... ")
    return {}

In [ ]:
# ---------------------------------------------------------------------------
# Node: generate_quiz - create a single quiz question from the summary alone
# ---------------------------------------------------------------------------
QUIZ_SYSTEM_PROMPT = (
    "You are HealthBot, creating a single comprehension-check question for a patient. "
    "You will be given a patient-friendly summary of a health topic. Write ONE clear, "
    "specific question that tests understanding of the summary.\n\n"
    "Rules:\n"
    "- The question MUST be answerable using only the information in the summary.\n"
    "- Do not introduce facts, numbers, or terms that are not in the summary.\n"
    "- Ask an open-ended question (not multiple choice, not yes/no).\n"
    "- Output ONLY the question text, nothing else."
)


def generate_quiz(state: HealthBotState) -> HealthBotState:
    prompt = f"Summary:\n{state['summary']}\n\nWrite the single quiz question now."
    response = llm.invoke([SystemMessage(content=QUIZ_SYSTEM_PROMPT), HumanMessage(content=prompt)])
    quiz_question = response.content

    return {"quiz_question": quiz_question, "messages": [response]}

In [ ]:
# ---------------------------------------------------------------------------
# Node: ask_quiz_question - present the question and collect the patient's answer
# ---------------------------------------------------------------------------
def ask_quiz_question(state: HealthBotState) -> HealthBotState:
    print("\n--- Comprehension Check ---")
    print(state["quiz_question"])
    answer = input("\nYour answer: ").strip()
    return {
        "patient_answer": answer,
        "messages": [HumanMessage(content=f"My answer: {answer}")],
    }

In [ ]:
# ---------------------------------------------------------------------------
# Node: grade_answer - grade the patient's answer using only the summary
# ---------------------------------------------------------------------------
GRADE_SYSTEM_PROMPT = (
    "You are HealthBot, grading a patient's answer to a comprehension-check question. "
    "You will be given the original summary, the quiz question, and the patient's answer.\n\n"
    "Rules:\n"
    "- Use ONLY the summary as your source of truth for what is correct.\n"
    "- Assign a letter grade (A, B, C, D, or F) reflecting how well the answer matches "
    "the summary's information.\n"
    "- Write a short justification (2-4 sentences) explaining the grade.\n"
    "- The justification MUST include at least one direct citation (a short quoted "
    "phrase or sentence) from the summary to reinforce the correct information.\n"
    "- Be encouraging and patient-friendly in tone, even when correcting mistakes.\n\n"
    "Respond in exactly this format:\n"
    "Grade: <letter>\n"
    "Justification: <your justification with citation(s)>"
)


def grade_answer(state: HealthBotState) -> HealthBotState:
    prompt = (
        f"Summary:\n{state['summary']}\n\n"
        f"Quiz question: {state['quiz_question']}\n\n"
        f"Patient's answer: {state['patient_answer']}\n\n"
        "Please grade this now."
    )
    response = llm.invoke([SystemMessage(content=GRADE_SYSTEM_PROMPT), HumanMessage(content=prompt)])
    result_text = response.content

    grade = ""
    feedback = result_text
    for line in result_text.splitlines():
        if line.lower().startswith("grade:"):
            grade = line.split(":", 1)[1].strip()
        elif line.lower().startswith("justification:"):
            feedback = line.split(":", 1)[1].strip()

    return {"grade": grade or "N/A", "feedback": feedback, "messages": [response]}

In [ ]:
# ---------------------------------------------------------------------------
# Node: present_grade - display the grade and feedback to the patient
# ---------------------------------------------------------------------------
def present_grade(state: HealthBotState) -> HealthBotState:
    print("\n--- Your Results ---")
    print(f"Grade: {state['grade']}")
    print(f"Feedback: {state['feedback']}")
    return {}

In [ ]:
# ---------------------------------------------------------------------------
# Node: ask_continue - ask whether to learn a new topic or exit
# ---------------------------------------------------------------------------
def ask_continue(state: HealthBotState) -> HealthBotState:
    choice = input("\nWould you like to learn about another health topic? (yes/no): ").strip().lower()
    return {"continue_session": choice.startswith("y")}


def route_continue(state: HealthBotState) -> str:
    return "reset_state" if state.get("continue_session") else END

In [ ]:
# ---------------------------------------------------------------------------
# Node: reset_state - clear topic-specific fields before starting a new topic
#
# This protects patient privacy/accuracy: the previous topic's summary, quiz,
# answer, and grade must NOT leak into the next topic's context.
# ---------------------------------------------------------------------------
def reset_state(state: HealthBotState) -> HealthBotState:
    print("\nStarting a fresh session for your new topic...\n")
    return {
        "messages": [RemoveMessage(id=m.id) for m in state["messages"]],
        "topic": None,
        "search_results": None,
        "summary": None,
        "quiz_question": None,
        "patient_answer": None,
        "grade": None,
        "feedback": None,
        "continue_session": None,
    }

The `reset_state` node needs `RemoveMessage` to clear the message history. Import it now:

In [ ]:
from langgraph.graph.message import RemoveMessage

## 7. Build the graph

The graph is cyclic: after the grade is presented, a conditional edge either loops back through `reset_state` to `ask_topic` (for a new topic, with a clean state) or ends the session.

In [ ]:
graph_builder = StateGraph(HealthBotState)

graph_builder.add_node("ask_topic", ask_topic)
graph_builder.add_node("search_topic", search_topic)
graph_builder.add_node("summarize_results", summarize_results)
graph_builder.add_node("present_summary", present_summary)
graph_builder.add_node("generate_quiz", generate_quiz)
graph_builder.add_node("ask_quiz_question", ask_quiz_question)
graph_builder.add_node("grade_answer", grade_answer)
graph_builder.add_node("present_grade", present_grade)
graph_builder.add_node("ask_continue", ask_continue)
graph_builder.add_node("reset_state", reset_state)

graph_builder.add_edge(START, "ask_topic")
graph_builder.add_edge("ask_topic", "search_topic")
graph_builder.add_edge("search_topic", "summarize_results")
graph_builder.add_edge("summarize_results", "present_summary")
graph_builder.add_edge("present_summary", "generate_quiz")
graph_builder.add_edge("generate_quiz", "ask_quiz_question")
graph_builder.add_edge("ask_quiz_question", "grade_answer")
graph_builder.add_edge("grade_answer", "present_grade")
graph_builder.add_edge("present_grade", "ask_continue")
graph_builder.add_conditional_edges(
    "ask_continue",
    route_continue,
    {"reset_state": "reset_state", END: END},
)
graph_builder.add_edge("reset_state", "ask_topic")

healthbot_graph = graph_builder.compile()

### Visualize the graph (optional)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(healthbot_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render graph image (this is optional):", e)
    print(healthbot_graph.get_graph().draw_mermaid())

## 8. Run the HealthBot

Running this cell starts an interactive session. You'll be prompted (via `input()`) for a health topic, then to continue after reading the summary, then for your quiz answer, and finally whether you'd like to explore another topic.

In [ ]:
initial_state: HealthBotState = {
    "messages": [],
    "topic": None,
    "search_results": None,
    "summary": None,
    "quiz_question": None,
    "patient_answer": None,
    "grade": None,
    "feedback": None,
    "continue_session": None,
}

final_state = healthbot_graph.invoke(initial_state, config={"recursion_limit": 100})
print("\nThanks for using HealthBot. Take care!")